# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)


In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

import pandas as pd  # noqa: E402
import numpy as np  # noqa: E402

pd.set_option("display.width", 120)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working dir:", os.getcwd())
print("Loaded:", df.shape)


Working dir: /root/FlyRank-Internship-ML
Loaded: (30000, 44)


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item (page)**, pseudonymized by `content_id`, belonging to one of
32 pseudonymized clients (`client_id`). There is no separate date column — every row is a single
**trailing-90-day snapshot ending at export time**, plus two nested 30-day comparison windows
(`*_last_30d` = most recent 30 days, `*_prev_30d` = the 30 days before that) used to compute the
trend fields. This is a cross-sectional snapshot, not a daily time series (the warehouse release
in notebook 03 is the time-series version, at `report_date x client x content` grain).

In [2]:
# Grain: one row per content_id, no duplicates
print("rows:", len(df), "| distinct content_id:", df["content_id"].nunique())
dup_counts = df.groupby("content_id").size()
print("content_id rows appearing more than once:", (dup_counts > 1).sum())

# Client coverage
print("distinct client_id:", df["client_id"].nunique())
print(df.groupby("client_id").size().describe())

# Window: content_age_days confirms every row has >=90 days of history at export time
print("\ncontent_age_days min/max:", df["content_age_days"].min(), df["content_age_days"].max())
print(df["age_tier"].value_counts())


rows: 30000 | distinct content_id: 30000
content_id rows appearing more than once: 0
distinct client_id: 32
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64

content_age_days min/max: 90 564
age_tier
91-180     11780
181-365    11368
365+        6360
31-90        492
Name: count, dtype: int64


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature** (knowable before the prediction moment, safe for a model):
  `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`,
  `word_count`, `char_count`, `content_age_days`, `age_tier`(_order), `days_since_last_update`,
  `freshness_tier`, `word_count_tier`, `char_count_tier`, and the 90d/30d activity counts
  (`impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`,
  `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`,
  `days_with_sessions`) together with their derived rates `ctr`, `avg_position`,
  `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier` —
  **but only the `*_prev_30d` slice of the 30-day pair**, since `*_last_30d` is one of the two
  inputs to the label (see below).

- **Label / proxy** — never a feature: `trend_direction` (label source: `is_declining_label =
  (trend_direction == "down")`) and `trend_pct` (computed from the same last-30-vs-prev-30
  comparison). `impressions_last_30d` / `clicks_last_30d` / `sessions_last_30d` are excluded
  from features for the same reason — they are half the label's own inputs.

- **Context** — for grouping/joining/splitting, never learned from: `content_id` (unique key),
  `client_id` (use for **client-holdout** splits — a page's client is knowable ahead of time,
  but splitting by client, not content_id, prevents a client's typical behaviour leaking between
  train and test).

- **Excluded**, each with a why:
  - `provider_used`, `model_used` — which LLM generated the article. Not a ranking-relevant
    property of the page's search performance, and 71.5% / 19.1% blank; keeping them risks the
    model keying off content-provenance instead of content quality.
  - `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` — label inputs (see above).


In [3]:
feature_cols = [
    "search_volume", "competition", "competition_level", "cpc", "content_type", "main_intent",
    "word_count", "char_count", "content_age_days", "age_tier", "age_tier_order",
    "days_since_last_update", "freshness_tier", "word_count_tier", "char_count_tier",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "impression_tier", "position_tier",
]
label_cols = ["trend_direction", "trend_pct"]
context_cols = ["content_id", "client_id"]
excluded_cols = ["provider_used", "model_used",
                  "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"]

all_cols = feature_cols + label_cols + context_cols + excluded_cols
print("classified:", len(all_cols), "/ total columns:", df.shape[1])
missing_from_contract = set(df.columns) - set(all_cols)
print("columns not yet classified above:", sorted(missing_from_contract))


classified: 44 / total columns: 44
columns not yet classified above: []


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Claims to check: grain holds, per-client counts are sane, missingness is systematic (not
random) and follows `content_type`, and the two windows behave as documented.

In [4]:
# Grain check: zero rows back means content_id is a true key
grain_check = df.groupby("content_id").size()
print("content_id groups with >1 row (should be 0):", (grain_check > 1).sum())

# Counts: rows per client, compare to the 32-client / 30,000-row claim
print("\ndistinct clients:", df["client_id"].nunique(), "| total rows:", len(df))

# Missingness: overall, and confirm it is patterned by content_type (not random)
print("\noverall missingness (nonzero columns):")
miss = df.isna().mean().sort_values(ascending=False)
print(miss[miss > 0].round(3))

print("\nkeyword-context missingness by content_type (search_volume as proxy):")
print(df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean()).round(3))

print("\nword_count missingness by content_type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()).round(3))

# Windows: content_age_days confirms every row has >=90 days of history (trailing-90d claim)
print("\ncontent_age_days min (should be >= 90):", df["content_age_days"].min())

# avg_position == 0 means "no data", not rank zero
print("avg_position == 0 rows (no position data):", (df["avg_position"] == 0).sum())

# trend_pct blank exactly when impressions_prev_30d == 0 (can't compute a % change from zero)
print("impressions_prev_30d == 0:", (df["impressions_prev_30d"] == 0).sum(),
      "| trend_pct isna:", df["trend_pct"].isna().sum())

# Rate columns that can exceed 100 (different measurement systems) — confirm it happens
print("scroll_rate > 100:", (df["scroll_rate"] > 100).sum(),
      "| ai_traffic_pct > 100:", (df["ai_traffic_pct"] > 100).sum())


content_id groups with >1 row (should be 0): 0

distinct clients: 32 | total rows: 30000

overall missingness (nonzero columns):


provider_used        0.715
char_count           0.257
word_count           0.257
word_count_tier      0.257
char_count_tier      0.257
model_used           0.191
trend_pct            0.113
competition_level    0.087
cpc                  0.082
competition          0.082
search_volume        0.082
main_intent          0.079
scroll_rate          0.004
dtype: float64

keyword-context missingness by content_type (search_volume as proxy):


content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: search_volume, dtype: float64

word_count missingness by content_type:
content_type
comparison article    0.000
feedly article        0.000
keyword article       0.283
Name: word_count, dtype: float64

content_age_days min (should be >= 90): 90
avg_position == 0 rows (no position data): 1205
impressions_prev_30d == 0: 3388 | trend_pct isna: 3388
scroll_rate > 100: 119 | ai_traffic_pct > 100: 23


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **No true time series.** This is one snapshot per page — it can show *that* a page trended
  down, never *when* day-by-day. Anything needing daily granularity (e.g. "did the drop start
  after a specific update") needs the warehouse release (notebook 03), not this file.
- **Provider/model fields are not usable as-is for anything client-facing.** Beyond being
  excluded from modeling, `provider_used` is blank 71.5% of the time — too sparse to describe
  the full slice.
- **`feedly article` rows carry zero keyword context** (`search_volume`, `competition`, `cpc`
  all 100% blank for that content_type) — a model trained across all three content types needs
  a `has_keyword_data` flag, not a `fillna(0)`, or it will silently learn "feedly article" from
  the zeros.
- **32 clients is a teaching slice, not the client population.** Rows per client range from 3 to
  7,008 (see the `describe()` above) — a handful of clients dominate the row count, so any
  aggregate stat computed without grouping by `client_id` is really a statement about the
  largest few clients.
- **`content_age_days` floor is 90** — this slice only contains pages already at or past the
  90-day mark (`age_tier` never shows `0-14` or `15-30`), so nothing here describes how brand-new
  content behaves.


In [5]:
print("rows per client — min/max spread:")
print(df.groupby("client_id").size().sort_values().iloc[[0, -1]])

print("\nage_tier values present in this slice (should exclude 0-14, 15-30):")
print(sorted(df["age_tier"].unique()))

print("\nfeedly article keyword-context blank rate (all three columns):")
feedly = df[df["content_type"] == "feedly article"]
print(feedly[["search_volume", "competition", "cpc"]].isna().mean())


rows per client — min/max spread:
client_id
client_1a6562590e       3
client_19581e27de    7008
dtype: int64

age_tier values present in this slice (should exclude 0-14, 15-30):
['181-365', '31-90', '365+', '91-180']

feedly article keyword-context blank rate (all three columns):
search_volume    1.0
competition      1.0
cpc              1.0
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.